In [1]:
import random
import math
from gdpc import Editor, Block, geometry

DIM = 30
LB, UB = -5.12, 5.12
POP = 60
ITER = 400
LIMIT = POP * DIM
rand = random.Random(8)
ACO_RAND = random.Random(123)

NON_SOLID = [
    "minecraft:cactus",
    "minecraft:bush",
    "minecraft:short_grass",
    "minecraft:tall_grass",
    "minecraft:fern",
    "minecraft:large_fern",
    "minecraft:dead_bush",
    "minecraft:short_dry_grass",
    "minecraft:tall_dry_grass",
    "minecraft:flowering_azalea",
    "minecraft:azalea",
    "minecraft:rose_bush",
    "minecraft:lilac",
    "minecraft:peony",
    "minecraft:sunflower",
    "minecraft:dandelion",
    "minecraft:poppy",
    "minecraft:blue_orchid",
    "minecraft:allium",
    "minecraft:azure_bluet",
    "minecraft:red_tulip",
    "minecraft:orange_tulip",
    "minecraft:white_tulip",
    "minecraft:pink_tulip",
    "minecraft:oxeye_daisy",
    "minecraft:cornflower",
    "minecraft:lily_of_the_valley",
    "minecraft:torchflower"
]

def clamp_to_buildarea(x, z, area):
    return (
        max(area.begin[0], min(x, area.end[0] - 1)),
        max(area.begin[2], min(z, area.end[2] - 1)),
    )


def get_surface_y(x, z, world_slice, rect):
    bx, bz = rect.begin
    w, d = rect.size
    ix = max(0, min(x - bx, w - 1))
    iz = max(0, min(z - bz, d - 1))
    return world_slice.heightmaps["WORLD_SURFACE"][ix][iz]


def vec_to_coords(vec, origin, area):
    w = area.end[0] - area.begin[0]
    d = area.end[2] - area.begin[2]
    coords = []
    for i in range(0, DIM, 2):
        tx = (vec[i] - LB) / (UB - LB)
        tz = (vec[i + 1] - LB) / (UB - LB)
        x = origin[0] + int(tx * w)
        z = origin[2] + int(tz * d)
        coords.append(clamp_to_buildarea(x, z, area))
    return coords


def settlement_fitness(vec, origin, area, world_slice, rect):
    coords = vec_to_coords(vec, origin, area)

    # spacing
    spacing = 0
    min_dist = 12
    for i in range(len(coords)):
        for j in range(i + 1, len(coords)):
            dx = coords[i][0] - coords[j][0]
            dz = coords[i][1] - coords[j][1]
            dist = (dx * dx + dz * dz) ** 0.5
            if dist < min_dist:
                spacing += (min_dist - dist) * 80

    # flatness
    flat = 0
    for x, z in coords:
        hs = []
        for dx in range(-2, 3):
            for dz in range(-2, 3):
                xx, zz = clamp_to_buildarea(x + dx, z + dz, area)
                hs.append(get_surface_y(xx, zz, world_slice, rect))
        flat += (max(hs) - min(hs)) * 20

    # water (3x3 around each)
    water = 0
    hmap = world_slice.heightmaps["WORLD_SURFACE"]
    bx, bz = rect.begin
    for x, z in coords:
        for dx in range(-1, 2):
            for dz in range(-1, 2):
                xx, zz = clamp_to_buildarea(x + dx, z + dz, area)
                ix = xx - bx
                iz = zz - bz
                y = hmap[ix][iz]
                bid = world_slice.getBlock((xx, y-2, zz)).id.lower()
                if "water" in bid:
                    water += 500
                    break

    # compactness
    cx = sum(x for x, _ in coords) / len(coords)
    cz = sum(z for _, z in coords) / len(coords)
    compact = sum(((x - cx) ** 2 + (z - cz) ** 2) ** 0.5 for x, z in coords)

    return spacing + flat + water + compact


def build_house(editor, x, y, z, size):
    h = 3
    geometry.placeCuboid(editor, (x-1, y, z-1), (x+size+1, y+h+1, z+size+1), Block("air"))
    # shell
    geometry.placeCuboid(editor, (x, y-1, z), (x+size, y+h, z+size), Block("oak_planks"))
    # hollow
    geometry.placeCuboid(editor, (x+1, y, z+1), (x+size-1, y+h, z+size-1), Block("air"))
    # roof
    geometry.placeCuboid(editor, (x, y+h+1, z), (x+size, y+h+1, z+size), Block("spruce_planks"))

    match random.randint(0, 3):
        case 0:  # south
            editor.placeBlock((x+size//2, y, z), Block("oak_door", {"facing": "south"}))
            return (x+size//2, z)
        case 1:  # west
            editor.placeBlock((x+size, y, z+size//2), Block("oak_door", {"facing": "west"}))
            return (x+size, z+size//2)
        case 2:  # north
            editor.placeBlock((x+size//2, y, z+size), Block("oak_door", {"facing": "north"}))
            return (x+size//2, z+size)
        case 3:  # east
            editor.placeBlock((x, y, z+size//2), Block("oak_door", {"facing": "east"}))
            return (x, z+size//2)


def house_mask(buildings, doors, size):
    blocked = set()
    for (x, z), (dx, dz) in zip(buildings, doors):
        for i in range(size + 1):
            for j in range(size + 1):
                blocked.add((x + i, z + j))
        blocked.add((dx, dz))
    return blocked


def distance(a, b):
    dx = a[0] - b[0]
    dz = a[1] - b[1]
    return math.hypot(dx, dz)


def carve_road(editor, a, b, area, world_slice, rect, blocked):
    path = Block("minecraft:dirt_path")
    x, z = a
    tx, tz = b
    mode = "direct"
    wall_dir = None
    dirs = [(1, 0), (0, 1), (-1, 0), (0, -1)]
    max_steps = 5000

    def is_blocked(px, pz):
        return (px, pz) in blocked

    for _ in range(max_steps):
        if (x, z) == (tx, tz):
            break

        if mode == "direct":
            dx = 1 if tx > x else -1 if tx < x else 0
            dz = 1 if tz > z else -1 if tz < z else 0
            step = (dx, 0) if abs(tx - x) >= abs(tz - z) else (0, dz)
            nx, nz = x + step[0], z + step[1]
            if is_blocked(nx, nz):
                mode = "wall"
                for d in dirs:
                    sx, sz = x + d[0], z + d[1]
                    if not is_blocked(sx, sz):
                        wall_dir = d
                        break
                continue
            x, z = nx, nz
        else:
            idx = dirs.index(wall_dir)
            right = dirs[(idx - 1) % 4]
            forward = wall_dir
            left = dirs[(idx + 1) % 4]

            for d in (right, forward, left):
                nx, nz = x + d[0], z + d[1]
                if not is_blocked(nx, nz):
                    wall_dir = d
                    x, z = nx, nz
                    break

            dx = 1 if tx > x else -1 if tx < x else 0
            dz = 1 if tz > z else -1 if tz < z else 0
            step = (dx, 0) if abs(tx - x) >= abs(tz - z) else (0, dz)
            nx, nz = x + step[0], z + step[1]
            if not is_blocked(nx, nz):
                mode = "direct"

        x, z = clamp_to_buildarea(x, z, area)
        y = get_surface_y(x, z, world_slice, rect)
        if world_slice.getBlock((x, y-1, z)).id.lower() in NON_SOLID: y-=1
        for dy in range(0, 3):
            editor.placeBlock((x, y + dy, z), Block("air"))
        editor.placeBlock((x, y - 1, z), path)

    if (x, z) == (tx, tz):
        y = get_surface_y(x, z, world_slice, rect)
        if world_slice.getBlock((x, y-1, z)).id.lower() in NON_SOLID: y-=1
        for dy in range(0, 3):
            editor.placeBlock((x, y + dy, z), Block("air"))
        editor.placeBlock((x, y - 1, z), path)


def aco_build_roads(editor, doors, area, world_slice, rect, blocked,
                    num_ants=15, iters=40, pheromone_weight=1.0, distance_weight=3.0, evaporation_rate=0.4, deposit_strength=100.0):
    n = len(doors)
    if n <= 1:
        return

    dist = [[0.0] * n for _ in range(n)]
    visibility = [[0.0] * n for _ in range(n)]
    for i in range(n):
        for j in range(n):
            if i == j:
                dist[i][j] = 1e9
                visibility[i][j] = 0.0
            else:
                d = distance(doors[i], doors[j])
                dist[i][j] = d
                visibility[i][j] = 1.0 / (d + 1e-6)

    pheromone = [[1.0] * n for _ in range(n)]
    best_tour, best_len = None, float("inf")

    for _ in range(iters):
        tours, lens = [], []
        for _a in range(num_ants):
            start = ACO_RAND.randrange(n)
            tour = [start]
            unvisited = set(range(n)) - {start}
            cur = start
            while unvisited:
                probs, denom = [], 0.0
                for j in unvisited:
                    val = (pheromone[cur][j] ** pheromone_weight) * (visibility[cur][j] ** distance_weight)
                    probs.append((j, val))
                    denom += val
                r = ACO_RAND.random() * denom
                acc, nxt = 0.0, None
                for j, val in probs:
                    acc += val
                    if acc >= r:
                        nxt = j
                        break
                if nxt is None:
                    nxt = next(iter(unvisited))
                tour.append(nxt)
                unvisited.remove(nxt)
                cur = nxt

            length = sum(dist[tour[i]][tour[i + 1]] for i in range(len(tour) - 1))
            tours.append(tour)
            lens.append(length)
            if length < best_len:
                best_len, best_tour = length, tour

        for i in range(n):
            for j in range(n):
                pheromone[i][j] = max(1e-6, pheromone[i][j] * (1.0 - evaporation_rate))

        for tour, length in zip(tours, lens):
            dep = deposit_strength / (length + 1e-6)
            for i in range(len(tour) - 1):
                a, b = tour[i], tour[i + 1]
                pheromone[a][b] += dep
                pheromone[b][a] += dep

    if best_tour:
        for i in range(len(best_tour) - 1):
            a = doors[best_tour[i]]
            b = doors[best_tour[i + 1]]
            carve_road(editor, a, b, area, world_slice, rect, blocked)


def abc_algorithm(world_slice, area, origin, rect):
    pop = [[LB + (UB - LB) * rand.random() for _ in range(DIM)] for _ in range(POP)]
    fit = [settlement_fitness(ind, origin, area, world_slice, rect) for ind in pop]
    trials = [0] * POP

    for _ in range(ITER):
        # employed
        for i in range(POP):
            k = rand.randrange(POP)
            while k == i:
                k = rand.randrange(POP)
            j = rand.randrange(DIM)
            phi = rand.random() * 2 - 1
            cand = pop[i].copy()
            cand[j] = max(LB, min(UB, pop[i][j] + phi * (pop[i][j] - pop[k][j])))
            f = settlement_fitness(cand, origin, area, world_slice, rect)
            if f < fit[i]:
                pop[i], fit[i], trials[i] = cand, f, 0
            else:
                trials[i] += 1

        # onlooker
        inv = [1.0 / (1.0 + v) for v in fit]
        mx = max(inv)
        prob = [0.9 * (v / mx) + 0.1 for v in inv]
        m, i = 0, 0
        while m < POP:
            if rand.random() < prob[i]:
                k = rand.randrange(POP)
                while k == i:
                    k = rand.randrange(POP)
                j = rand.randrange(DIM)
                phi = rand.random() * 2 - 1
                cand = pop[i].copy()
                cand[j] = max(LB, min(UB, pop[i][j] + phi * (pop[i][j] - pop[k][j])))
                f = settlement_fitness(cand, origin, area, world_slice, rect)
                if f < fit[i]:
                    pop[i], fit[i], trials[i] = cand, f, 0
                else:
                    trials[i] += 1
                m += 1
            i = (i + 1) % POP

        # scout
        worst = max(range(POP), key=lambda idx: trials[idx])
        if trials[worst] >= LIMIT:
            pop[worst] = [LB + (UB - LB) * rand.random() for _ in range(DIM)]
            fit[worst] = settlement_fitness(pop[worst], origin, area, world_slice, rect)
            trials[worst] = 0

    best = min(range(POP), key=lambda idx: fit[idx])
    return pop[best]


def generate_settlement():
    editor = Editor(buffering=True, host="http://localhost:8080")
    area = editor.getBuildArea()
    rect = area.toRect()
    world_slice = editor.loadWorldSlice(rect, cache=True)
    origin = (area.begin[0], area.begin[1], area.begin[2])

    best_vec = abc_algorithm(world_slice, area, origin, rect)
    buildings = vec_to_coords(best_vec, origin, area)

    doors = []
    for x, z in buildings:
        x, z = clamp_to_buildarea(x, z, area)
        y = get_surface_y(x, z, world_slice, rect)
        doors.append(build_house(editor, x, y, z, size=6))

    blocked = house_mask(buildings, doors, size=6)
    aco_build_roads(editor, doors, area, world_slice, rect, blocked)
    editor.flushBuffer()


generate_settlement()
